In [ ]:
import os
import base64
from pathlib import Path
from email.mime.text import MIMEText

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/gmail.send"]
TOKEN_FILE = Path("gmail_token.json")

client_config = {
    "installed": {
        "client_id": os.environ["fallmailclient_id"],
        "client_secret": os.environ["fallmailclient_pass"],
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "redirect_uris": ["http://localhost"],
    }
}

creds = None
if TOKEN_FILE.exists():
    creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
        creds = flow.run_local_server(port=0, open_browser=True)

    TOKEN_FILE.write_text(creds.to_json(), encoding="utf-8")

gmail = build("gmail", "v1", credentials=creds)

recipient = "kamesh29kumar@gmail.com"
message = MIMEText("Hi I am K.A.R.U.P and this is a test email")
message["To"] = recipient
message["Subject"] = "Gmail API test"

response = gmail.users().messages().send(
    userId="me",
    body={"raw": base64.urlsafe_b64encode(message.as_bytes()).decode()},
).execute()

print(f"Sent successfully — message ID: {response['id']}")

Sent successfully — message ID: 1a03e05aaea29087
